In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from utils import *
import prompt
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import pickle


# LIBD

In [ ]:
results_dict = {}

# Define directories and column name
deconv_result_dir = "examples/visium_libd/deconv_result/"
spatial_data_dir = str(dataset_root("visium_libd"))
name_truth = "layer_guess"

# Get list of sample folders
sample_folders = [f for f in os.listdir(spatial_data_dir) if os.path.isdir(os.path.join(spatial_data_dir, f))]
sample_folders.sort()

In [ ]:



# Process each sample
for data_name in sample_folders:
    config = load_config("configs/config_cluster_deconv.yaml")
    config.data_name = data_name
    config.refresh_paths()
    name_truth = config.name_truth

    # --- Load data ---
    data_path = f"{spatial_data_dir}/{config.data_name}/"
    # marker_path = "data/reference/marker_genes/Mouse_cell_markers.txt"
    adata = sc.read_visium(data_path)
    adata.var_names_make_unique()

    # Normalize data
    sc.pp.filter_genes(adata, min_cells=10)
    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)
    sc.pp.scale(adata)
    

    cell_proportion_data = pd.read_csv(os.path.join(deconv_result_dir, f"celltype_proportions_{config.data_name}.csv"), index_col=0)

    adata.obs = adata.obs.join(cell_proportion_data)

    # read the metadata
    meta_data = pd.read_csv(os.path.join(data_path, "metadata.tsv"), sep="\t")
    # merge the metadata to adata
    adata.obs = adata.obs.merge(meta_data, left_index=True, right_index=True, how="left")

    # Remove rows with NaN values in 'layer_guess'
    adata = adata[~adata.obs[name_truth].isna()].copy()

    # Verify that NaNs have been removed
    remaining_nan_count = adata.obs[name_truth].isna().sum()
    print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

    # Rename the obs_names of adata
    adata.obs_names = [f'spot_{i}' for i in range(len(adata.obs_names))]

    # Transform spatial coordinates to DataFrame for sparse_adjacency
    pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
    cell_proportion_data = adata.obs[cell_proportion_data.columns].copy()

    important_marker_genes = ["Aqp4", "Hpcal1", "Pvalb", "Frem3", "Pcp4", "Krt17","Mobp", 
                          "Lamp5", "Rorb", "Fezf2", "Syt6", "Fa2h",
                          "Plp1", "Foxj1", "Gfap", "Cpne5", "Kcnip2",
                          "Bgn", "Cux2", "Etv1",
                          "Zmat4", "Rab3c"]
    top_genes = [gene.upper() for gene in important_marker_genes]
    top_genes = list(set(adata.var_names) & set(top_genes))

    kmeansBoth_ari = []
    kmeans_ari = []
    kemans_genes_ari = []
    kmeansBoth_nmi = []
    kmeans_nmi = []
    kemans_genes_nmi = []

    for r in range(10, 2000, 10):
        adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
        # add diagonal to the adj_matrix
        adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

        # Calculate the number of neighbors of each node
        n_neighbors = adj_matrix.sum(axis=1).mean()
        print(f"Number of neighbors: {sig_figs(n_neighbors, 3)}")

        # If you want to store the n_neighbors of each node
        n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
        adata.obs['n_neighbors'] = n_neighbors

        # --- Calculate neighbor counts ---
        neighbor_count = adj_matrix.dot(cell_proportion_data)

        # Convert n_neighbors to a column vector for element-wise division
        n_neighbors_col = n_neighbors.reshape(-1, 1)
        # Perform element-wise division between neighbor_count and n_neighbors_col
        neighbor_matrix_normalized = neighbor_count / n_neighbors_col

        # transform the neighbor matrix to a dataframe
        neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized, index=cell_proportion_data.index, 
                                    columns=cell_proportion_data.columns)
        
        # --- Calculate neighbor genes ---
        neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

        # Perform element-wise division between neighbor_count and n_neighbors_col
        neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

        neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                                    index=adata.obs_names, 
                                    columns=top_genes)
        
        neighbor_scaled_df = neighbor_normalized_df.join(neighbor_normalized_df_genes).copy()
        
        # kmeans with both neighbor count and neighbor genes
        km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
        clusters = km.fit_predict(neighbor_scaled_df)
        kmeansBoth_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
        kmeansBoth_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))

        # kmeans with neighbor count
        km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
        clusters = km.fit_predict(neighbor_normalized_df)
        kmeans_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
        kmeans_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
        # kmeans with neighbor genes
        km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
        clusters = km.fit_predict(neighbor_normalized_df_genes)
        kemans_genes_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
        kemans_genes_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
    results_dict[data_name] = {"kmeans_ari": kmeans_ari, "kmeans_genes_ari": kemans_genes_ari, "kmeansBoth_ari": kmeansBoth_ari,
                              "kmeans_nmi": kmeans_nmi, "kmeans_genes_nmi": kemans_genes_nmi, "kmeansBoth_nmi": kmeansBoth_nmi}




In [ ]:
# plot results_dict in 12 subplots in one figure
fig, axs = plt.subplots(3, 4, figsize=(20, 15))
for i, data_name in enumerate(sample_folders):
    axs[i//4, i%4].plot(range(10, 2000, 10), results_dict[data_name]["kmeansBoth_ari"], label="kmeansBoth")
    axs[i//4, i%4].plot(range(10, 2000, 10), results_dict[data_name]["kmeans_ari"], label="kmeans")
    axs[i//4, i%4].plot(range(10, 2000, 10), results_dict[data_name]["kmeans_genes_ari"], label="kmeans_genes")
    axs[i//4, i%4].legend()
    axs[i//4, i%4].set_title(data_name)
plt.show()

In [ ]:
# plot nmi
fig, axs = plt.subplots(3, 4, figsize=(20, 15))
for i, data_name in enumerate(sample_folders):
    axs[i//4, i%4].plot(range(10, 2000, 10), results_dict[data_name]["kmeansBoth_nmi"], label="kmeansBoth")
    axs[i//4, i%4].plot(range(10, 2000, 10), results_dict[data_name]["kmeans_nmi"], label="kmeans")
    axs[i//4, i%4].plot(range(10, 2000, 10), results_dict[data_name]["kmeans_genes_nmi"], label="kmeans_genes")
    axs[i//4, i%4].legend()
    axs[i//4, i%4].set_title(data_name)
plt.show()

In [ ]:
# save the results_dict
with open('examples/results/kmeans_results/kmeans_r_spot_results_dict.pkl', 'wb') as f:
    pickle.dump(results_dict, f)

In [ ]:
# read the results_dict
with open('examples/results/kmeans_results/kmeans_r_spot_results_dict.pkl', 'rb') as f:
    results_dict = pickle.load(f)


In [ ]:
# Collect NMI scores at r=500
r500_index = range(10, 2000, 10).index(500)  # Find index corresponding to r=500
kmeansBoth_r500 = []
kmeans_r500 = []
kmeans_genes_r500 = []

for data_name in sample_folders:
    kmeansBoth_r500.append(results_dict[data_name]["kmeansBoth_nmi"][r500_index])
    kmeans_r500.append(results_dict[data_name]["kmeans_nmi"][r500_index])
    kmeans_genes_r500.append(results_dict[data_name]["kmeans_genes_nmi"][r500_index])

# Create boxplot
plt.figure(figsize=(8, 6))
box_data = [kmeansBoth_r500, kmeans_r500, kmeans_genes_r500]
plt.boxplot(box_data, labels=['KMeans Both', 'KMeans', 'KMeans Genes'])
plt.title('NMI Scores at r=500 Across Datasets')
plt.ylabel('NMI Score')
plt.grid(True, alpha=0.3)

# Add individual points for each dataset
for i, data in enumerate(box_data, 1):
    plt.scatter([i] * len(data), data, alpha=0.6, color='red', 
                marker='o', s=50, zorder=3)

plt.show()

In [ ]:
results_dict["151673"]["kmeansBoth_nmi"][r500_index]
